# NB9 — follow-up report

Pairs every follow-up arm with its baseline by seed and dev item: dev accuracy on the common grid, paired group-bootstrap contrasts at update 35, at the final update and at the dev-selected checkpoint (the better of the two), early learning speed from the training losses, the gradient spectrum and run costs. CPU only, a few minutes.

**Accelerator:** None · **Internet:** OFF · **Attach:** pcl-next code, NB2, all NB3, all NB4, NB7, all NB8 outputs

Run with **Save Version → Save & Run All (Commit)**. Afterwards: read `report_next/report.md`.

In [ ]:
# --- Setup (identical in every notebook; nothing to edit) --------------------
import json, os, shutil, subprocess, sys, time
from pathlib import Path
NOTEBOOK_STARTED = time.time()
KAGGLE = Path(os.environ.get('PCL_KAGGLE_ROOT', '/kaggle'))   # override only for a local rehearsal
INPUT, WORKING = KAGGLE / 'input', KAGGLE / 'working'

code_roots = sorted({p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'pcl' / 'cli.py').is_file()})
assert len(code_roots) == 1, f'attach exactly one copy of the pcl code dataset; found {code_roots}'
WORK = Path(os.environ.get('PCL_WORK', '/tmp/protected-cl'))
if not WORK.exists():
    shutil.copytree(code_roots[0], WORK, ignore=shutil.ignore_patterns('outputs', '__pycache__', '*.egg-info', '.pytest_cache'))
sys.path.insert(0, str(WORK))   # `python -m pcl` runs from WORK, so no package install is needed

import importlib.metadata as metadata
def _version(name):
    try:
        return tuple(int(x) for x in metadata.version(name).split('.')[:2])
    except Exception:
        return (0,)
needed = {'transformers': (4, 51), 'safetensors': (0, 4), 'accelerate': (1, 0), 'PyYAML': (6, 0)}
missing = [f'{n}>={".".join(map(str, v))}' for n, v in needed.items() if _version(n) < v]
if missing:
    print('Installing', missing, '(needs Internet ON for this notebook)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import yaml
from pcl import kaggle
from pcl.io import read_json

def pcl(*args):
    """Run one pcl command in a fresh process (its GPU memory is released when it exits)."""
    print('$ pcl', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pcl', *map(str, args)], check=True, cwd=WORK)

def on_gpus(commands, log_dir):
    """One pcl command per T4, logs streamed live; fails if either command fails."""
    commands = [json.dumps([sys.executable, '-m', 'pcl', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

def show(path):
    try:
        from IPython.display import Markdown, display
        display(Markdown(Path(path).read_text()))
    except ImportError:
        print(Path(path).read_text())

def remaining_seconds():
    """Session time left, from SESSION_HOURS in the first cell."""
    return int(SESSION_HOURS * 3600 - (time.time() - NOTEBOOK_STARTED))

found = kaggle.discover(INPUT, exclude=code_roots[0])
print({key: len(value) for key, value in found.items()})
# --- Follow-up add-on (pclx lives in notebooks_2/ of the same code dataset) ------------
NEXT = WORK / 'notebooks_2'
assert (NEXT / 'pclx' / 'cli.py').is_file(), 'attach the pcl-next code dataset (it contains notebooks_2/)'
sys.path.insert(0, str(NEXT))
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in (str(WORK), str(NEXT), os.environ.get('PYTHONPATH', '')) if p)
from pclx import plan

def pclx(*args):
    print('$ pclx', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pclx', *map(str, args)], check=True, cwd=WORK)

def pclx_on_gpus(commands, log_dir):
    """One pclx command per T4 (at most two), logs streamed live."""
    commands = [json.dumps([sys.executable, '-m', 'pclx', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

found_next = plan.discover_next([INPUT, WORKING], exclude=code_roots[0])
print({'stage_a_seeds': plan.stage_a_seeds(found_next), 'spectra': sorted(found_next['spectra']),
       'finished_items': sorted(k for k in found_next['items'] if plan.item_complete(found_next, *k))})

In [ ]:
pclx('report', '--search', INPUT, '--exclude', code_roots[0], '--out', WORKING / 'report_next')
show(WORKING / 'report_next' / 'report.md')
try:
    from IPython.display import Image, display
    for figure in sorted((WORKING / 'report_next').glob('*.png')):
        display(Image(filename=str(figure)))
except ImportError:
    pass